# CV language & religion augmentation (Cohere)

Set `COHERE_API_KEY` in your environment before running (e.g. in a terminal: `export COHERE_API_KEY=...`, or use `os.environ["COHERE_API_KEY"] = "..."` in the next cell for local use only).

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import cohere

_api_key = os.environ.get("COHERE_API_KEY")
if not _api_key:
    raise RuntimeError("Set COHERE_API_KEY in your environment.")
co = cohere.Client(_api_key)

_cwd = Path.cwd()
SCRIPT_DIR = _cwd if (_cwd / "paths.py").is_file() else _cwd / "dataset creation scripts"
if SCRIPT_DIR.is_dir() and (SCRIPT_DIR / "paths.py").is_file():
    if str(SCRIPT_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPT_DIR))

from paths import DATA_DIR
from constants import AUGMENTED_RESUME_COLS

import lang_religion_augmentation
lang_religion_augmentation.init_client(co)
from lang_religion_augmentation import (
    SYSTEM_INSTRUCTIONS,
    RESUME_COLS,
    add_language_and_religion,
    find_unaugmented_resume_slots,
    retry_failed_resume_slots,
    rerun_api_for_section_mismatch_report,
)

## Dataset construction pipeline

All scripts live in **`dataset creation scripts/`**; CSV data goes to **`data/`** (see `paths.DATA_DIR`). Shared constants (model name, column names) are in `constants.py`. Order:

1. **`rephrase_resume_variants.py`** — two lexical rephrases per row (`resume` -> `resume 2 `, `resume 3`).
2. **`generate_study_df_repositioned.py`** — expand 105 rows -> 315 (`permutation_id`, `baseline_at_pos`, `markers`, rotated texts).
3. **`jd_placeholders_augmentation.py`** — fill JD placeholders; sets `city_type`, `assigned_city`.
4. **`placeholder_augmentation.py`** — fill CV placeholders + `[PLZ]` patch.
5. **`lang_religion_augmentation.py`** (this notebook) — `add_language_and_religion` -> `added_lang_and_rel_cvs.csv`.
6. **`build_cv_analysis_long.py`** — optional wide -> long table for analysis.

QC: **`compare_cv_sections.py`** checks section headings survived augmentation.

The next cell imports the other pipeline helpers; uncomment calls when you run a step.

In [ ]:
from rephrase_resume_variants import rephrase_resumes_two_versions
from generate_study_df_repositioned import generate_study_df_repositioned
from jd_placeholders_augmentation import (
    SYSTEM_INSTRUCTIONS as JD_PLACEHOLDER_INSTRUCTIONS,
    augment_jds_final_version,
)
from placeholder_augmentation import (
    SYSTEM_INSTRUCTIONS as CV_PLACEHOLDER_INSTRUCTIONS,
    run_placeholder_augmentation,
)
from build_cv_analysis_long import build_from_csv

# --- Example calls (uncomment; use your real paths / dataframe names) ---
# rephrase_resumes_two_versions(df_105, client=co)
# markers_df = generate_study_df_repositioned(df_105)
# jd_aug_df = augment_jds_final_version(cleaned_df, JD_PLACEHOLDER_INSTRUCTIONS, client=co)
# jd_aug_df.to_csv(DATA_DIR / "fully_augmented_cv_dataset.csv", index=False)
# cv_ph_df = run_placeholder_augmentation(markers_df, CV_PLACEHOLDER_INSTRUCTIONS, client=co)
# build_from_csv(DATA_DIR / "added_lang_and_rel_cvs.csv", DATA_DIR / "final_CV_dataset_long.csv")


## Load data

Expects **`data/augmented_cvs_full.csv`** and **`data/fully_augmented_cv_dataset.csv`** (see `paths.DATA_DIR`). After load, `jd` on the CV-side frame is dropped and replaced by `augmented_job_description`, `city_type`, and `assigned_city` from the merged file (on `resume_id`, `permutation_id`, `baseline_at_pos`).

In [12]:
pre_language_addition_df = pd.read_csv(DATA_DIR / "augmented_cvs_full.csv")
augmented_job_descriptions_df = pd.read_csv(
    DATA_DIR / "fully_augmented_cv_dataset.csv"
)

_merge_keys = ["resume_id", "permutation_id", "baseline_at_pos"]
_from_aug = augmented_job_descriptions_df[
    _merge_keys + ["jd", "city_type", "assigned_city"]
].rename(columns={"jd": "augmented_job_description"})
pre_language_addition_df = pre_language_addition_df.drop(columns=["jd"])
pre_language_addition_df = pre_language_addition_df.merge(
    _from_aug, on=_merge_keys, how="left"
)

pre_language_addition_df.columns, augmented_job_descriptions_df.columns

(Index(['resume_id', 'placeholders', 'jd_placeholders', 'permutation_id',
        'baseline_at_pos', 'markers', 'augmented_resume_1',
        'augmented_resume_2', 'augmented_resume_3', 'augmented_job_description',
        'city_type', 'assigned_city'],
       dtype='str'),
 Index(['resume_id', 'placeholders', 'jd_placeholders', 'jd', 'permutation_id',
        'baseline_at_pos', 'markers', 'augmented_resume_1',
        'augmented_resume_2', 'augmented_resume_3', 'city_type',
        'assigned_city'],
       dtype='str'))

## Run augmentation

Writes **`data/added_lang_and_rel_cvs.csv`** and **`data/checkpoint_added_lang_and_rel_cvs.csv`** (see `DATA_DIR`).

1. **Pilot (5 rows):** `add_language_and_religion(..., max_rows=5, resume=False)` — inspect `added_lang_and_rel_cvs.csv`.
2. **Continue:** `add_language_and_religion(..., resume=True)` — loads that CSV (or `checkpoint_...` if needed), skips rows with the same `resume_id` / `permutation_id` / `baseline_at_pos`, runs the rest, saves the combined file (sorted by those ids).

In [13]:
len(pre_language_addition_df)


315

In [23]:
# Pilot: first 5 rows only (inspect added_lang_and_rel_cvs.csv, then use the next line)
#all_markers_df = add_language_and_religion(
#    pre_language_addition_df, SYSTEM_INSTRUCTIONS, max_rows=5, resume=False
#)

# Full run: continue from existing output / checkpoint (omit max_rows)
all_markers_df = add_language_and_religion(
    pre_language_addition_df, SYSTEM_INSTRUCTIONS, resume=True
)

Resume: loaded 5 rows from 'added_lang_and_rel_cvs.csv'
Already saved: 5 | Still to run this batch: 310 (of 315 in input)
API pass: 310 row(s) this run, up to 3 parallel calls per row | rows already in file: 5
[78/310]  25.2% this pass | index 82 | total rows 83

Error for column augmented_resume_1: The read operation timed out


[83/310]  26.8% this pass | index 87 | total rows 88

Error for column augmented_resume_3: The read operation timed out


[101/310]  32.6% this pass | index 105 | total rows 106

...

[246/310]  79.4% this pass | index 250 | total rows 251

Error for column augmented_resume_1: The read operation timed out


[260/310]  83.9% this pass | index 264 | total rows 265

Error for column augmented_resume_3: The read operation timed out


[310/310] 100.0% this pass | index 314 | total rows 315
Done. Saved to: added_lang_and_rel_cvs.csv


## Find slots that likely need API retry

Run **Load data** and the cell that defines `find_unaugmented_resume_slots` first. The code below prefers **`all_markers_df`** from the augmentation run; if that variable is missing (e.g. new kernel), it loads `added_lang_and_rel_cvs.csv`. Read-only — no API calls.

**Older logs** only showed `df_index=…`. After **Load data**, map with: `pre_language_addition_df.loc[82, "resume_id"]` (replace `82` with the logged index). New runs print **`resume_id`** on progress lines and API errors (`permutation_id` / `baseline_at_pos` stay in the data for merges and retries, but are omitted from those messages).


In [34]:

_augmented = pd.read_csv(str(DATA_DIR / "added_lang_and_rel_cvs.csv"))

_slots_to_retry = find_unaugmented_resume_slots(_augmented, pre_language_addition_df)
print(f"{len(_slots_to_retry)} slot(s) where output still matches input but markers expect an edit")
# Full ``_slots_to_retry`` (incl. permutation_id, baseline_at_pos) is required for ``retry_failed_resume_slots``
_slots_to_retry[["df_index", "resume_id", "resume_column"]]


0 slot(s) where output still matches input but markers expect an edit


,df_index,resume_id,resume_column


## Retry API for finder matches

Run **Load data**, the **helpers + retry** cell, and the **finder** cell first (so `_augmented` matches your CSV). `retry_failed_resume_slots` **re-runs the finder internally** and calls Cohere **only** for those row×column slots (sequential, longer timeout). Afterwards run the **finder** cell again — it should report 0 slots.


In [32]:
try:
    _augmented = all_markers_df
except NameError:
    _augmented = pd.read_csv(str(DATA_DIR / "added_lang_and_rel_cvs.csv"))

all_markers_df = retry_failed_resume_slots(
    _augmented,
    pre_language_addition_df,
    SYSTEM_INSTRUCTIONS,
)


Retrying 6 slot(s); HTTP timeout up to 180s per attempt
[1/6] OK df_index=183 resume_id=1269 augmented_resume_1
[2/6] OK df_index=82 resume_id=1270 augmented_resume_1
[3/6] OK df_index=250 resume_id=1356 augmented_resume_1


Error for column augmented_resume_2: The read operation timed out


[4/6] OK df_index=142 resume_id=1369 augmented_resume_2
[5/6] OK df_index=264 resume_id=1273 augmented_resume_3
[6/6] OK df_index=106 resume_id=1355 augmented_resume_3
Saved: added_lang_and_rel_cvs.csv


## CV section completeness check

Uses `compare_cv_sections.py` next to this notebook: compares **original vs augmented** resume text for **section headings** (heuristic). Headings are normalized for markdown (`#`, `**…**`, `*…*`) so one side bold and the other plain should not count as missing. **Language** sections (Sprachen, …) are **not** required in the augmented text. Run **Load data** first. Augmented side: `all_markers_df` if present, else `added_lang_and_rel_cvs.csv`. Report columns: `missing_sections_readable`, `missing_canonical_sections`; writes `cv_section_mismatch_report.csv`.


In [46]:
import importlib
import compare_cv_sections
importlib.reload(compare_cv_sections)
from compare_cv_sections import compare_dataframes

try:
    _orig = pre_language_addition_df
except NameError:
    raise RuntimeError(
        "Run the Load data cell first so pre_language_addition_df exists"
    )

_aug = pd.read_csv(str(DATA_DIR / "added_lang_and_rel_cvs.csv"))

_cv_section_report = compare_dataframes(_orig, _aug)
_cv_section_report.to_csv(str(DATA_DIR / "cv_section_mismatch_report.csv"), index=False)
print(
    len(_cv_section_report),
    "slot(s) with missing non-language sections -> cv_section_mismatch_report.csv",
)
_cols = [
    "aug_df_index",
    "resume_id",
    "permutation_id",
    "baseline_at_pos",
    "resume_column",
    "n_missing",
    "missing_sections_readable",
    "missing_canonical_sections",
]
_disp = [c for c in _cols if c in _cv_section_report.columns]
_miss = set(_cols) - set(_disp)
if _miss:
    print("Warning: report missing columns (re-run helpers after saving compare_cv_sections.py):", _miss)
_cv_section_report[_disp] if len(_cv_section_report) else _cv_section_report


2 slot(s) with missing non-language sections -> cv_section_mismatch_report.csv


,aug_df_index,resume_id,permutation_id,baseline_at_pos,resume_column,n_missing,missing_sections_readable,missing_canonical_sections
0,174,1333,1,1,augmented_resume_1,1,Interests / Hobbys,interests
1,175,1333,2,3,augmented_resume_3,1,Interests / Hobbys,interests


## Optional dataset quality checks

Run after **Load data** (and optionally load `added_lang_and_rel_cvs.csv` as `_aug_q`). These complement the **section mismatch** report.

| Check | What it catches |
|--------|------------------|
| **Row & key counts** | Wrong N vs 315; duplicate `(resume_id, permutation_id, baseline_at_pos)` |
| **JD merge** | NaNs in `augmented_job_description` / `city_type` / `assigned_city` after left-merge |
| **Resume text** | Empty cells; very short strings (truncation); extreme outliers in length |
| **Markers** | Unparseable blocks; counts of `language:` / `religion:` lines |
| **Design balance** | `permutation_id` and `baseline_at_pos` distributions (spot skew) |
| **Aug vs input** | Rows where augmented text still equals pre-aug (possible API skip) — use `find_unaugmented_resume_slots` with your slot rules |
| **Spot audit** | Random `resume_id` + column: read raw text for coherence |

The next cell runs a small automated subset; extend the DataFrames as needed.


In [47]:
# --- Quick quality report (extend / comment out as needed) ---
_keys = ["resume_id", "permutation_id", "baseline_at_pos"]

try:
    _df_in = pre_language_addition_df
except NameError:
    raise RuntimeError("Run Load data first")

try:
    _df_out = pd.read_csv(str(DATA_DIR / "added_lang_and_rel_cvs.csv"))
except Exception as e:
    print("No augmented CSV:", e)
    _df_out = None

print("=== Input frame ===")
print("rows:", len(_df_in), "| dup keys:", _df_in.duplicated(_keys).sum())
for c in ("augmented_job_description", "city_type", "assigned_city"):
    if c in _df_in.columns:
        print(f"  {c} NaN: {_df_in[c].isna().sum()}")

if _df_out is not None:
    print("\n=== Output CSV ===")
    print("rows:", len(_df_out), "| dup keys:", _df_out.duplicated(_keys).sum())
    for c in RESUME_COLS:
        empty = _df_out[c].fillna("").astype(str).str.len().eq(0).sum()
        lens = _df_out[c].fillna("").astype(str).str.len()
        print(
            f"  {c}: empty={empty} | len min/median/max = {lens.min()}/{lens.median():.0f}/{lens.max()}"
        )
    if "permutation_id" in _df_out.columns:
        print("  permutation_id:", _df_out["permutation_id"].value_counts().sort_index().to_dict())

print("\n=== Section mismatch file (if present) ===")
import os

if os.path.isfile(str(DATA_DIR / "cv_section_mismatch_report.csv")):
    _r = pd.read_csv(str(DATA_DIR / "cv_section_mismatch_report.csv"))
    print("rows:", len(_r))
else:
    print("(run section check cell first)")


=== Input frame ===
rows: 315 | dup keys: 0
  augmented_job_description NaN: 0
  city_type NaN: 0
  assigned_city NaN: 0

=== Output CSV ===
rows: 315 | dup keys: 0
  augmented_resume_1: empty=0 | len min/median/max = 1384/3447/8675
  augmented_resume_2: empty=0 | len min/median/max = 1363/3446/8826
  augmented_resume_3: empty=0 | len min/median/max = 1709/3473/8932
  permutation_id: {1: 105, 2: 105, 3: 105}

=== Section mismatch file (if present) ===
rows: 2


## Re-run API for section-mismatch report

After **`cv_section_mismatch_report.csv`** / `_cv_section_report`, call **`rerun_api_for_section_mismatch_report`** to send those `(resume_id, permutation_id, baseline_at_pos) × resume_column` slots through Cohere again. By default the model receives **pre-augmentation** text from `pre_language_addition_df` (not the broken augmented cell). Pass **`exclude_slots`** to skip rows you have manually verified. Requires the **retry** cell and **Load data**. Then re-run the **section check**.


In [45]:
_aug_fix = pd.read_csv(str(DATA_DIR / "added_lang_and_rel_cvs.csv"))

_section_rerun_exclude = pd.DataFrame(
    [
        {
            "resume_id": 1333,
            "permutation_id": 1,
            "baseline_at_pos": 1,
            "resume_column": "augmented_resume_1",
        },
        {
            "resume_id": 1333,
            "permutation_id": 2,
            "baseline_at_pos": 3,
            "resume_column": "augmented_resume_3",
        },
    ]
)

all_markers_df = rerun_api_for_section_mismatch_report(
    str(DATA_DIR / "cv_section_mismatch_report.csv"),
    _aug_fix,
    pre_language_addition_df,
    SYSTEM_INSTRUCTIONS,
    use_input_df_text=True,
    exclude_slots=_section_rerun_exclude,
)


Excluded 2 slot(s) from re-run (exclude_slots)
Section-mismatch re-run: 9 unique slot(s); timeout 240s; source text = input_df (pre-aug)
[1/9] OK resume_id=1282 augmented_resume_3
[2/9] OK resume_id=1289 augmented_resume_2
[3/9] OK resume_id=1296 augmented_resume_2
[4/9] OK resume_id=1330 augmented_resume_3
[5/9] OK resume_id=1349 augmented_resume_2
[6/9] OK resume_id=1368 augmented_resume_2
[7/9] OK resume_id=1369 augmented_resume_2
[8/9] OK resume_id=1375 augmented_resume_3
[9/9] OK resume_id=1379 augmented_resume_1
Saved: added_lang_and_rel_cvs.csv
